In [1]:
from pathlib import Path
import os

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "run_flowette.py").exists():
    raise FileNotFoundError(
        "Open this notebook from the extracted project directory, or set PROJECT_ROOT manually."
    )
os.chdir(PROJECT_ROOT)

In [2]:
TASK = "guacamol" # Choose from qm9, zinc, moses, guacamol

# Set SMOKE_RUN=True only to verify the installation quickly.
SMOKE_RUN = False
FORCE_RETRAIN = True

if SMOKE_RUN:
    MAX_GRAPHS = 5000
    AE_EPOCHS = 10
    FM_EPOCHS = 50
    NUM_SAMPLES = 200
    SKIP_EVAL = True
else:
    MAX_GRAPHS = None
    AE_EPOCHS = 50
    FM_EPOCHS = 500
    NUM_SAMPLES = 1000
    SKIP_EVAL = False

print({
    "task": TASK,
    "max_graphs": MAX_GRAPHS,
    "ae_epochs": AE_EPOCHS,
    "fm_epochs": FM_EPOCHS,
    "samples": NUM_SAMPLES,
})

In [1]:
from run_flowette import main as run_flowette

argv = [
    "--task", TASK,
    "--max-num-graphs", str(MAX_GRAPHS),
    "--ae-epochs", str(AE_EPOCHS),
    "--fm-epochs", str(FM_EPOCHS),
    "--fm-layers", "5",
    "--pairing-strategy", "auto",
    "--pairing-refresh-every", "5",
    "--n-steps", "200",
    "--num-samples", str(NUM_SAMPLES),
]
if FORCE_RETRAIN:
    argv.append("--force-train-fm")
if SKIP_EVAL:
    argv.append("--skip-eval")

generated_molecules, metrics = run_flowette(argv)

In [3]:
from rdkit import Chem

valid_smiles = [
    Chem.MolToSmiles(m, canonical=True, isomericSmiles=True)
    for m in generated_molecules
    if m is not None
]
print(f"Valid decoded molecules: {len(valid_smiles)}/{len(generated_molecules)}")
print("First 200 SMILES:")
for smi in valid_smiles[:200]:
    print(smi)
print("Metrics:", metrics)

In [4]:
import os
import pandas as pd
from IPython.display import display
from rdkit import Chem
from rdkit.Chem import Draw
from rdkit.Chem.Draw import IPythonConsole

IPythonConsole.drawOptions.maxMols = 1000

valid_mols = [m for m in generated_molecules if m is not None]
valid_smiles = [Chem.MolToSmiles(m, canonical=True, isomericSmiles=True) for m in valid_mols]

subset_mols = valid_mols[:200]
subset_smiles = valid_smiles[:200]
legends = [f"{i}\n{smi}" for i, smi in enumerate(subset_smiles)]

img = Draw.MolsToGridImage(
    subset_mols,
    molsPerRow=10,
    subImgSize=(200, 200),
    legends=legends,
    maxMols=1000,
    useSVG=False,
    returnPNG=False,
)

os.makedirs("images", exist_ok=True)
timestamp = pd.Timestamp.now().strftime("%Y%m%d_%H%M%S")

out_path = f"images/{TASK}_generated_molecules_{timestamp}.png"

img.save(out_path)
print(f"Saved image at -> {out_path}")
display(img)